# Project: Parallel Report Generator

This notebook accompanies the [Agent Foundry](https://agent-foundry-pi.vercel.app) LangGraph roadmap.

You will build a report generator using the orchestrator-worker pattern: an orchestrator plans sections with structured output, distributes writing to parallel workers via the Send API, and a synthesizer aggregates the final report.

## 1. Install Dependencies

In [ ]:
!pip install -q langgraph langchain "langchain[openai]"

## 2. Set Up Your API Key

Enter your OpenAI API key when prompted. The key is not stored or displayed.

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Define State and Structured Output Models

The report state uses `Annotated[list[str], operator.add]` so parallel workers can all append their sections.

In [ ]:
import operator
from typing import TypedDict, Annotated
from pydantic import BaseModel

class Section(BaseModel):
    title: str
    description: str

class ReportPlan(BaseModel):
    title: str
    sections: list[Section]

class ReportState(TypedDict):
    topic: str
    plan: dict
    sections: Annotated[list[str], operator.add]
    report: str

class SectionState(TypedDict):
    title: str
    description: str

print("ReportState fields:", list(ReportState.__annotations__))
print("SectionState fields:", list(SectionState.__annotations__))

## 4. Build the Orchestrator

Uses structured output to plan report sections dynamically.

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

def orchestrator(state: ReportState) -> dict:
    plan = llm.with_structured_output(ReportPlan).invoke(
        f"Create a report plan with 3-5 sections about: {state['topic']}. "
        f"Each section needs a title and a one-sentence description of what to cover."
    )
    return {"plan": plan.model_dump()}

print("Orchestrator defined")

## 5. Dynamic Distribution with Send

A conditional edge reads the plan and creates one `Send` per section.

In [ ]:
from langgraph.types import Send

def distribute_sections(state: ReportState) -> list[Send]:
    sections = state["plan"]["sections"]
    return [
        Send("writer", {"title": s["title"], "description": s["description"]})
        for s in sections
    ]

print("Distribution function defined")

## 6. Writer and Synthesizer Nodes

In [ ]:
def writer(state: SectionState) -> dict:
    content = llm.invoke(
        f"Write a detailed section for a report.\n"
        f"Section title: {state['title']}\n"
        f"Description: {state['description']}\n"
        f"Write 2-3 paragraphs. Start with the section title as a markdown heading."
    )
    return {"sections": [content.content]}

def synthesizer(state: ReportState) -> dict:
    all_sections = "\n\n".join(state["sections"])
    report = llm.invoke(
        f"You are given individual sections of a report about '{state['topic']}'.\n"
        f"Combine them into a cohesive final report. Add an introduction and conclusion.\n"
        f"Sections:\n\n{all_sections}"
    )
    return {"report": report.content}

print("Writer and synthesizer defined")

## 7. Assemble and Compile the Graph

In [ ]:
from langgraph.graph import StateGraph, START, END

graph = StateGraph(ReportState)

graph.add_node("orchestrator", orchestrator)
graph.add_node("writer", writer)
graph.add_node("synthesizer", synthesizer)

graph.add_edge(START, "orchestrator")
graph.add_conditional_edges("orchestrator", distribute_sections, path_map=["writer"])
graph.add_edge("writer", "synthesizer")
graph.add_edge("synthesizer", END)

app = graph.compile()
print("Report generator graph compiled")

## 8. Generate a Report

In [ ]:
result = app.invoke({"topic": "The Impact of Large Language Models on Software Engineering"})

print(f"Report plan: {result['plan']['title']}")
print(f"Sections planned: {len(result['plan']['sections'])}")
for s in result["plan"]["sections"]:
    print(f"  - {s['title']}")

print(f"\nSections generated: {len(result['sections'])}")

In [ ]:
print("=== FINAL REPORT ===")
print(result["report"])

## 9. Try a Different Topic

In [ ]:
result2 = app.invoke({"topic": "The Future of Autonomous Vehicles"})

print(f"Sections planned: {len(result2['plan']['sections'])}")
for s in result2["plan"]["sections"]:
    print(f"  - {s['title']}")

print(f"\n=== REPORT ===\n{result2['report'][:500]}...")

## What You Learned

1. The **orchestrator-worker pattern** splits into plan, parallel execute, and synthesize stages
2. **Structured output** (`with_structured_output`) produces typed plans for dynamic section creation
3. **`Send` API** creates one worker per section — the count is determined at runtime
4. **`Annotated[list[str], operator.add]`** aggregates all parallel worker outputs into a single list
5. The **synthesizer** receives all sections and produces a cohesive final report